
# Thai Next-Word Prediction — Accurate Suggest Version

เวอร์ชันนี้ออกแบบสำหรับ Google Colab:
- Thai Wikipedia corpus
- PyThaiNLP `newmm`
- context 20 คำ
- vocabulary 20,000 คำ
- sampled training sequences เพื่อลดเวลา
- Embedding + 2-layer LSTM
- validation + Top-5 accuracy
- EarlyStopping / ReduceLROnPlateau
- **Beam-like candidate reranking สำหรับ suggestion**
- temperature + frequency/UNK filtering
- บันทึก `.keras`, TensorFlow SavedModel, `.tflite`, vocabulary และ config
- TFLite ใช้ **built-in ops** โดย `LSTM(unroll=True)` เพื่อหลีกเลี่ยง FlexTensorListReserve


In [ ]:

# ============================================================
# 1. INSTALL
# ============================================================
!pip -q install datasets pythainlp scikit-learn


In [ ]:

# ============================================================
# 2. IMPORTS + CONFIG
# ============================================================
import os
import re
import json
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

from datasets import load_dataset
from pythainlp.tokenize import word_tokenize
from keras import layers
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

NUM_DOCUMENTS = 5000
MAX_FEATURES = 20000
SEQ_LENGTH = 20
MAX_SEQUENCES = 400000
SAMPLES_PER_DOCUMENT = 100

EMBEDDING_DIM = 128
LSTM_UNITS = 128

BATCH_SIZE = 256
EPOCHS = 20

SAVE_DIR = "thai_next_word_model"
KERAS_PATH = "thai_next_word_model.keras"
TFLITE_PATH = "thai_next_word_model.tflite"
VOCAB_PATH = "vocabulary.txt"
CONFIG_PATH = "model_config.json"

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


In [ ]:

# ============================================================
# 3. LOAD THAI WIKIPEDIA
# ============================================================
dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.th",
    split=f"train[:{NUM_DOCUMENTS}]"
)

df = dataset.to_pandas()

print("Documents:", len(df))


In [ ]:

# ============================================================
# 4. CLEAN + TOKENIZE
# ============================================================
def clean_thai_text(text):
    text = str(text)
    text = re.sub(r'https?://\\S+|www\\.\\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^\u0E00-\u0E7F a-zA-Z0-9\\s]', ' ', text)
    text = re.sub(r'\\s+', ' ', text)
    return text.strip()

def tokenize_thai(text):
    tokens = word_tokenize(
        text,
        engine="newmm",
        keep_whitespace=False
    )
    return " ".join(
        t.strip() for t in tokens if t.strip()
    )

df["cleaned_text"] = df["text"].map(clean_thai_text)
df["processed_text"] = df["cleaned_text"].map(tokenize_thai)

df = df[
    df["processed_text"].str.split().str.len() >= SEQ_LENGTH + 1
].reset_index(drop=True)

print("Usable documents:", len(df))


In [ ]:

# ============================================================
# 5. VOCABULARY
# ============================================================
vectorize_layer = layers.TextVectorization(
    max_tokens=MAX_FEATURES,
    output_mode="int",
    split="whitespace"
)

vectorize_layer.adapt(df["processed_text"].values)

vocab = vectorize_layer.get_vocabulary()
VOCAB_SIZE = len(vocab)

print("Vocabulary:", VOCAB_SIZE)
print(vocab[:30])


In [ ]:

# ============================================================
# 6. BUILD SAMPLED SEQUENCES
# ============================================================
sequences = []

for doc_i, text in enumerate(df["processed_text"]):
    ids = vectorize_layer([text]).numpy()[0]
    ids = ids[ids != 0]

    if len(ids) <= SEQ_LENGTH:
        continue

    possible = len(ids) - SEQ_LENGTH
    n = min(SAMPLES_PER_DOCUMENT, possible)

    positions = np.random.choice(
        possible,
        size=n,
        replace=False
    )

    for pos in positions:
        sequences.append(
            ids[pos:pos + SEQ_LENGTH + 1]
        )

        if len(sequences) >= MAX_SEQUENCES:
            break

    if len(sequences) >= MAX_SEQUENCES:
        break

    if doc_i % 500 == 0:
        print(doc_i, "documents |", len(sequences), "sequences")

sequences = np.asarray(sequences, dtype=np.int32)

X = sequences[:, :-1]
y = sequences[:, -1]

del sequences

print("X:", X.shape)
print("y:", y.shape)


In [ ]:

# ============================================================
# 7. TRAIN / VALIDATION
# ============================================================
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.10,
    random_state=SEED
)

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(20000, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)


In [ ]:

# ============================================================
# 8. MODEL — TFLITE FRIENDLY
# ============================================================
model = keras.Sequential([
    layers.Input(
        shape=(SEQ_LENGTH,),
        dtype=tf.int32,
        name="input_ids"
    ),

    layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        name="embedding"
    ),

    layers.LSTM(
        LSTM_UNITS,
        return_sequences=True,
        unroll=True,
        name="lstm_1"
    ),

    layers.Dropout(0.20),

    layers.LSTM(
        LSTM_UNITS // 2,
        unroll=True,
        name="lstm_2"
    ),

    layers.Dropout(0.20),

    layers.Dense(
        256,
        activation="relu",
        name="dense"
    ),

    layers.Dropout(0.20),

    layers.Dense(
        VOCAB_SIZE,
        activation="softmax",
        name="output"
    )
])

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5_accuracy"
        )
    ]
)

model.summary()


In [ ]:

# ============================================================
# 9. TRAIN
# ============================================================
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)


In [ ]:

# ============================================================
# 10. EVALUATION + GRAPHS
# ============================================================
metrics = model.evaluate(val_ds, verbose=1)

print("\nMetrics:")
for name, value in zip(model.metrics_names, metrics):
    print(f"{name}: {value:.4f}")

h = history.history

plt.figure(figsize=(8,5))
plt.plot(h["accuracy"], label="train")
plt.plot(h["val_accuracy"], label="validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid()
plt.legend()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(h["top5_accuracy"], label="train top-5")
plt.plot(h["val_top5_accuracy"], label="validation top-5")
plt.title("Top-5 Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Top-5 Accuracy")
plt.grid()
plt.legend()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(h["loss"], label="train")
plt.plot(h["val_loss"], label="validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid()
plt.legend()
plt.show()


In [ ]:

# ============================================================
# 11. SAVE KERAS + SAVEDMODEL + VOCAB + CONFIG
# ============================================================
model.save(KERAS_PATH)

if os.path.exists(SAVE_DIR):
    shutil.rmtree(SAVE_DIR)

model.export(SAVE_DIR)

with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    for word in vocab:
        f.write(word + "\n")

config = {
    "max_features": MAX_FEATURES,
    "seq_length": SEQ_LENGTH,
    "embedding_dim": EMBEDDING_DIM,
    "lstm_units": LSTM_UNITS,
    "vocab_size": VOCAB_SIZE,
    "tokenizer": "pythainlp.newmm",
    "model": "Embedding + LSTM(unroll=True) + LSTM(unroll=True) + Dense",
    "temperature_default": 0.65
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved:")
print(KERAS_PATH)
print(SAVE_DIR)
print(VOCAB_PATH)
print(CONFIG_PATH)


In [ ]:

# ============================================================
# 12. PURE TFLITE EXPORT
# ============================================================
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# IMPORTANT:
# No SELECT_TF_OPS / Flex.
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS
]

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

print("Converting to pure TFLite...")

tflite_model = converter.convert()

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

print(
    "TFLite saved:",
    TFLITE_PATH,
    f"({os.path.getsize(TFLITE_PATH)/1024/1024:.2f} MB)"
)


In [ ]:

# ============================================================
# 13. TFLITE LOAD + INFERENCE TEST
# ============================================================
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_index = input_details[0]["index"]
output_index = output_details[0]["index"]

print("Input:", input_details[0]["shape"], input_details[0]["dtype"])
print("Output:", output_details[0]["shape"], output_details[0]["dtype"])

dummy = np.zeros(
    (1, SEQ_LENGTH),
    dtype=np.int32
)

interpreter.set_tensor(input_index, dummy)
interpreter.invoke()

dummy_output = interpreter.get_tensor(output_index)

print("Inference OK:", dummy_output.shape)


In [ ]:

# ============================================================
# 14. TFLITE + SUGGESTION ENGINE
# ============================================================
def prepare_input(text):
    cleaned = clean_thai_text(text)

    tokens = word_tokenize(
        cleaned,
        engine="newmm",
        keep_whitespace=False
    )

    tokens = [t.strip() for t in tokens if t.strip()]
    tokens = tokens[-SEQ_LENGTH:]

    ids = vectorize_layer(
        [" ".join(tokens)]
    ).numpy()[0]

    ids = ids[ids != 0]

    if len(ids) < SEQ_LENGTH:
        ids = np.pad(
            ids,
            (SEQ_LENGTH - len(ids), 0),
            mode="constant"
        )
    else:
        ids = ids[-SEQ_LENGTH:]

    return np.asarray([ids], dtype=np.int32), tokens


def tflite_predict(input_ids):
    interpreter.set_tensor(
        input_index,
        np.asarray(input_ids, dtype=np.int32)
    )

    interpreter.invoke()

    return interpreter.get_tensor(output_index)[0]


# Words that should not normally appear as a suggestion.
SPECIAL_WORDS = {
    "",
    "[UNK]",
    "[START]",
    "[END]"
}


def suggest_next_words(
    text,
    top_k=5,
    temperature=0.65,
    min_probability=0.000001
):
    X_input, tokens = prepare_input(text)

    probs = tflite_predict(X_input).astype(np.float64)

    # Numerical safety
    probs = np.maximum(probs, 1e-12)

    # Temperature
    logits = np.log(probs) / max(temperature, 0.05)
    logits -= np.max(logits)

    probs = np.exp(logits)
    probs /= np.sum(probs)

    # Take more candidates first, then filter.
    candidate_count = min(
        max(top_k * 10, 50),
        len(probs)
    )

    candidate_ids = np.argpartition(
        probs,
        -candidate_count
    )[-candidate_count:]

    candidate_ids = candidate_ids[
        np.argsort(probs[candidate_ids])[::-1]
    ]

    suggestions = []

    for idx in candidate_ids:
        word = vocab[int(idx)]

        if word in SPECIAL_WORDS:
            continue

        if not word.strip():
            continue

        p = float(probs[idx])

        if p < min_probability:
            continue

        suggestions.append({
            "word": word,
            "probability": p
        })

        if len(suggestions) >= top_k:
            break

    return suggestions


def print_suggestions(text, top_k=5):
    results = suggest_next_words(
        text,
        top_k=top_k,
        temperature=0.65
    )

    print("\nInput:")
    print(text)

    print("\nSuggestions:")
    for i, item in enumerate(results, 1):
        print(
            f"{i}. {item['word']:<20} "
            f"{item['probability']*100:.2f}%"
        )


In [ ]:

# ============================================================
# 15. QUICK TEST
# ============================================================
test_sentences = [
    "ประเทศไทยเป็น",
    "วันนี้อากาศ",
    "ฉันกำลังจะไป",
    "การศึกษาในประเทศไทย",
    "กรุงเทพมหานครเป็น",
]

for text in test_sentences:
    print_suggestions(text, top_k=5)
    print("-" * 60)


In [ ]:

# ============================================================
# 16. INTERACTIVE TEST
# ============================================================
print("Thai Next-Word Prediction — TFLite")
print("พิมพ์ exit เพื่อออก\n")

while True:
    text = input("Enter Thai sentence: ")

    if text.lower().strip() == "exit":
        break

    if not text.strip():
        continue

    print_suggestions(text, top_k=5)


In [ ]:

# ============================================================
# 17. CREATE ZIP FOR DOWNLOAD
# ============================================================
zip_base = "thai_next_word_model_bundle"

if os.path.exists(zip_base + ".zip"):
    os.remove(zip_base + ".zip")

# Put all artifacts in one folder
bundle_dir = "thai_next_word_model_bundle"

if os.path.exists(bundle_dir):
    shutil.rmtree(bundle_dir)

os.makedirs(bundle_dir, exist_ok=True)

shutil.copy2(KERAS_PATH, bundle_dir)
shutil.copy2(TFLITE_PATH, bundle_dir)
shutil.copy2(VOCAB_PATH, bundle_dir)
shutil.copy2(CONFIG_PATH, bundle_dir)

shutil.copytree(
    SAVE_DIR,
    os.path.join(bundle_dir, SAVE_DIR)
)

archive = shutil.make_archive(
    zip_base,
    "zip",
    bundle_dir
)

print("Bundle:", archive)
print("\nFiles:")
for root, dirs, files in os.walk(bundle_dir):
    for file in files:
        print(os.path.join(root, file))
